# SonicSentinel AI – dataset exploration & features
Run from the project root after `scripts/build_dataset.py`. Shows class balance, durations, quality, an example waveform/spectrogram and the feature vector.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from config.settings import DATASET_METADATA_CSV, BASE_DIR, CLASSES
meta = pd.read_csv(DATASET_METADATA_CSV)
meta.head()

In [ ]:
orig = meta[meta.is_augmented == 0]
pd.crosstab(orig.class_label, orig.split).reindex(CLASSES)

In [ ]:
orig.groupby('class_label').duration.describe().reindex(CLASSES)

In [ ]:
orig.quality.value_counts().plot.bar(title='Audio quality of original clips'); plt.show()

In [ ]:
from audio_preprocessing import load_audio, preprocess_signal, segment, analyze_quality
from feature_extraction import extract_features, feature_names
import librosa, librosa.display
row = orig[orig.class_label == 'Gunshot'].iloc[0]
a = load_audio(BASE_DIR / row.path)
clean, _ = preprocess_signal(a.samples, a.sample_rate)
fig, ax = plt.subplots(2, 1, figsize=(10, 5))
librosa.display.waveshow(clean, sr=22050, ax=ax[0])
S = librosa.power_to_db(librosa.feature.melspectrogram(y=clean, sr=22050), ref=np.max)
librosa.display.specshow(S, sr=22050, x_axis='time', y_axis='mel', ax=ax[1]); plt.tight_layout()
analyze_quality(a.samples, a.sample_rate)

In [ ]:
segs = segment(clean, 22050, 2.0, 1.0)
X = np.vstack([extract_features(s) for _, _, s in segs])
pd.DataFrame(X, columns=feature_names()).iloc[:, :12]

## Model results
After `python -m python_models.train_models`:

In [ ]:
pd.read_csv(BASE_DIR / 'reports' / 'python_model_comparison.csv')

In [ ]:
from IPython.display import Image
Image(str(BASE_DIR / 'reports' / 'confusion_matrix_python.png'))